# Density Estimation and Feature Discovery — From First Principles

_Generated: 2025-10-22T18:07:20.163153Z_

This notebook builds **density estimators** and **feature discovery** methods from scratch using NumPy:

• **Kernel Density Estimation (KDE)** in 1D and 2D with Gaussian kernels, including **bandwidth selection** via cross‑validated log‑likelihood.
• **Gaussian Mixture Model (GMM)** learned by the **EM algorithm**, with **BIC** for model selection.
• **PCA** (via SVD) for linear feature extraction and variance analysis.
• **ICA** (FastICA‑style) for independent latent factor discovery.

We use a **synthetic 2D mixture** for density visualization and the **Wine** dataset for feature discovery.


## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib scikit-learn

## 1) Imports & Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

np.set_printoptions(precision=4, suppress=True)
SEED = 7
rng = np.random.default_rng(SEED)

def standardize(X, eps=1e-12):
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < eps, 1.0, sd)
    return (X - mu)/sd, mu, sd

def cov_diag(X):
    return X.var(axis=0, ddof=1)

## 2) Data: Synthetic 2D Mixture for Density + Wine Dataset for Features

In [ ]:
def make_synth_2d(n=600, rng=rng):
    mix = rng.choice([0,1,2], size=n, p=[0.4, 0.4, 0.2])
    means = np.array([[0,0], [3,2], [-3,2.5]], dtype=float)
    covs = np.array([[[1.0, 0.2],[0.2, 0.6]], [[0.6,-0.1],[-0.1,0.8]], [[0.4,0.0],[0.0,0.7]]])
    X = np.zeros((n,2))
    for i,m in enumerate(mix):
        X[i] = rng.multivariate_normal(means[m], covs[m])
    return X

X2 = make_synth_2d(700, rng)
X2s, mu2, sd2 = standardize(X2)

wine = datasets.load_wine()
Xw = wine.data.astype(float); yw = wine.target
Xws, muw, sdw = standardize(Xw)
print("Synthetic 2D:", X2.shape, "| Wine:", Xw.shape, "classes:", np.unique(yw).size)

## 3) Kernel Density Estimation (1D and 2D) — from scratch

In [ ]:
def gaussian_kernel(u):
    return np.exp(-0.5*u*u)/np.sqrt(2*np.pi)

def kde_1d_fit(X):
    return X.ravel().astype(float)

def kde_1d_pdf(model, x, h):
    X = model
    n = len(X)
    x = np.atleast_1d(x).astype(float)
    U = (x[None,:] - X[:,None]) / h
    K = gaussian_kernel(U)
    return np.mean(K, axis=0) / h

def silverman_bandwidth_1d(X):
    X = np.ravel(X).astype(float)
    n = len(X); s = np.std(X, ddof=1)
    iqr = np.subtract(*np.percentile(X, [75, 25]))
    a = min(s, iqr/1.349) if iqr>0 else s
    return 0.9*a*(n**(-1/5))

def cv_loglik_kde_1d(X, h_values):
    X = np.ravel(X).astype(float)
    n = len(X)
    ll = []
    for h in h_values:
        U = (X[:,None] - X[None,:]) / h
        K = gaussian_kernel(U)
        diag = np.diag_indices(n)
        K[diag] = 0.0
        denom = (np.sum(K, axis=1) / (n-1)) / h + 1e-300
        ll.append(np.mean(np.log(denom)))
    return np.array(ll)

x1 = X2s[:,0]
h0 = silverman_bandwidth_1d(x1)
grid = np.linspace(x1.min()-3, x1.max()+3, 400)
h_candidates = np.linspace(0.2*h0, 2.5*h0, 25)
ll = cv_loglik_kde_1d(x1, h_candidates)
h_best = float(h_candidates[np.argmax(ll)])

model1d = kde_1d_fit(x1)
p_grid = kde_1d_pdf(model1d, grid, h_best)

fig = plt.figure(figsize=(7,4))
plt.plot(grid, p_grid, label=f"KDE 1D (h={h_best:.3f})")
plt.hist(x1, bins=30, density=True, alpha=0.3, label="data hist")
plt.title("1D KDE on standardized x₁"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
def kde_2d_pdf(X, points, h):
    # X: (n,2) standardized; points: (m,2); bandwidth scalar h (isotropic)
    X = np.asarray(X, float)
    P = np.asarray(points, float)
    n = X.shape[0]
    D2 = np.sum((P[:,None,:] - X[None,:,:])**2, axis=2)
    val = np.exp(-0.5*D2/(h*h)).sum(axis=1) / (n * (2*np.pi*(h*h)))
    return val

n = len(X2s); d = 2
h2 = (n**(-1.0/(d+4)))  # isotropic bandwidth on standardized data
xx, yy = np.meshgrid(np.linspace(X2s[:,0].min()-3, X2s[:,0].max()+3, 150),
                     np.linspace(X2s[:,1].min()-3, X2s[:,1].max()+3, 150))
pts = np.c_[xx.ravel(), yy.ravel()]
den = kde_2d_pdf(X2s, pts, h=h2).reshape(xx.shape)

fig = plt.figure(figsize=(6,5))
plt.contourf(xx, yy, den, levels=20)
plt.scatter(X2s[:,0], X2s[:,1], s=8, alpha=0.5, c='k')
plt.title(f"2D KDE (isotropic) with h≈{h2:.3f}"); plt.tight_layout(); plt.show()

## 4) Gaussian Mixture Model via EM (from scratch) + BIC

In [ ]:
def gmm_em(X, k=3, max_iter=200, tol=1e-6, seed=SEED, n_init=5, reg=1e-6):
    X = np.asarray(X, float)
    n, d = X.shape
    best = None
    best_ll = -np.inf
    rng_local = np.random.default_rng(seed)
    for run in range(n_init):
        mu = X[rng_local.choice(n, size=k, replace=False)].copy()
        Sigma = np.array([np.cov(X.T) + reg*np.eye(d) for _ in range(k)])
        pi = np.ones(k)/k
        ll_hist = []
        for it in range(max_iter):
            log_comp = np.zeros((n,k))
            for j in range(k):
                Sj = Sigma[j]
                det = np.linalg.det(Sj) + 1e-300
                inv = np.linalg.inv(Sj)
                diff = X - mu[j]
                quad = np.sum(diff @ inv * diff, axis=1)
                log_comp[:,j] = np.log(pi[j]+1e-300) - 0.5*(d*np.log(2*np.pi) + np.log(det) + quad)
            m = np.max(log_comp, axis=1, keepdims=True)
            resp = np.exp(log_comp - m)
            sum_resp = np.sum(resp, axis=1, keepdims=True) + 1e-300
            gamma = resp / sum_resp
            ll = float(np.sum(m + np.log(sum_resp)))
            ll_hist.append(ll)
            if it>0 and abs(ll_hist[-1] - ll_hist[-2]) < tol:
                break
            Nk = gamma.sum(axis=0) + 1e-300
            pi = Nk / n
            mu = (gamma.T @ X) / Nk[:,None]
            for j in range(k):
                diff = X - mu[j]
                Sigma[j] = (diff.T @ (diff * gamma[:,j][:,None])) / Nk[j] + reg*np.eye(d)
        if ll > best_ll:
            best_ll = ll
            best = {"pi":pi, "mu":mu, "Sigma":Sigma, "ll_history":ll_hist, "k":k}
    return best

def gmm_bic(model, X):
    n, d = X.shape
    k = model["k"]
    p = (k-1) + k*d + k*(d*(d+1))//2
    logL = model["ll_history"][-1]
    return -2*logL + p*np.log(n)

bic_vals = []; models = []
for kk in [1,2,3,4,5]:
    m = gmm_em(X2s, k=kk, n_init=5, max_iter=200, tol=1e-4, reg=1e-5)
    b = gmm_bic(m, X2s)
    bic_vals.append(b); models.append(m)
    print(f"k={kk}  logL={m['ll_history'][-1]:.2f}  BIC={b:.2f}")
k_best = int(np.argmin(bic_vals)+1)
best_gmm = models[np.argmin(bic_vals)]
print("Selected k* by BIC:", k_best)

mu = best_gmm["mu"]; Sigma = best_gmm["Sigma"]; pi = best_gmm["pi"]
xx, yy = np.meshgrid(np.linspace(X2s[:,0].min()-3, X2s[:,0].max()+3, 200),
                     np.linspace(X2s[:,1].min()-3, X2s[:,1].max()+3, 200))
P = np.c_[xx.ravel(), yy.ravel()]
def mvn_pdf(P, m, S):
    d = P.shape[1]
    inv = np.linalg.inv(S); det = np.linalg.det(S)+1e-300
    diff = P - m
    quad = np.sum(diff @ inv * diff, axis=1)
    return np.exp(-0.5*(quad + d*np.log(2*np.pi) + np.log(det)))
mix = sum(pi[j]*mvn_pdf(P, mu[j], Sigma[j]) for j in range(k_best)).reshape(xx.shape)

fig = plt.figure(figsize=(6,5))
plt.contourf(xx, yy, mix, levels=20)
plt.scatter(X2s[:,0], X2s[:,1], s=8, alpha=0.5, c='k')
plt.scatter(mu[:,0], mu[:,1], marker='X', s=120, c='white')
plt.title(f"GMM-EM density (k*={k_best} by BIC)"); plt.tight_layout(); plt.show()

fig = plt.figure(figsize=(6,3.5))
plt.plot(bic_vals, marker='o'); plt.xticks(range(len(bic_vals)), [1,2,3,4,5])
plt.xlabel("k"); plt.ylabel("BIC"); plt.title("GMM model selection by BIC")
plt.tight_layout(); plt.show()

## 5) PCA from scratch (via SVD) on Wine

In [ ]:
def pca_svd(X, n_components=None):
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    comps = Vt
    explained_var = (S**2) / (len(X)-1)
    evr = explained_var / explained_var.sum()
    if n_components is not None:
        return Xc @ comps[:n_components].T, comps[:n_components], evr[:n_components], evr
    return Xc @ comps.T, comps, evr, evr

Zw, Cw, evr_k, evr_all = pca_svd(Xws, n_components=2)
cum_evr = np.cumsum(evr_all)

fig = plt.figure(figsize=(6,3.8))
plt.plot(np.arange(1,len(evr_all)+1), cum_evr, marker='o')
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.title("PCA on Wine (standardized)"); plt.tight_layout(); plt.show()

fig = plt.figure(figsize=(6,5))
plt.scatter(Zw[:,0], Zw[:,1], c=yw, s=20, cmap='viridis')
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Wine projected to first 2 PCs")
plt.tight_layout(); plt.show()

def top_loadings(C, feature_names, k=5):
    for i in range(min(2, C.shape[0])):
        w = C[i]
        order = np.argsort(np.abs(w))[::-1][:k]
        print(f"PC{i+1} top {k} loadings:")
        for j in order:
            print(f"  {feature_names[j]:25s} {w[j]:+.3f}")
top_loadings(Cw, wine.feature_names, k=6)

## 6) ICA (FastICA-style) on Wine — from scratch

In [ ]:
def fastica(X, n_components, max_iter=500, tol=1e-5, seed=SEED):
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    Xw = Xc @ Vt[:n_components].T
    Xw /= S[:n_components]
    rng_local = np.random.default_rng(seed)
    W = rng_local.normal(0,1,size=(n_components, n_components))
    def decorrelate(W):
        U2, S2, V2 = np.linalg.svd(W, full_matrices=False)
        return (U2 @ V2)
    W = decorrelate(W)
    for it in range(max_iter):
        Y = Xw @ W.T
        gY = np.tanh(Y)
        gYp = 1 - gY**2
        W_new = (gY.T @ Xw)/Xw.shape[0] - np.diag(gYp.mean(axis=0)) * W
        W_new = decorrelate(W_new)
        if np.max(np.abs(np.abs(np.diag(W_new @ W.T)) - 1.0)) < tol:
            W = W_new; break
        W = W_new
    S_est = Xw @ W.T
    A_est = np.linalg.pinv(W)
    return S_est, W, A_est

Sica, Wica, Aica = fastica(Xws, n_components=5, max_iter=800, tol=1e-6, seed=SEED)
fig = plt.figure(figsize=(7,4))
for i in range(5):
    plt.plot(Sica[:200,i] + 3*i, label=f"IC{i+1}")
plt.title("First 5 Independent Components (first 200 samples, offset)")
plt.tight_layout(); plt.show()

print("ICA component directions (approx, top contributing features):")
def pca_components_all(X):
    Xc = X - X.mean(axis=0, keepdims=True)
    U,S,Vt = np.linalg.svd(Xc, full_matrices=False)
    return Vt
C_all = pca_components_all(Xws)
feature_dir = C_all[:Sica.shape[1]].T @ Wica.T
for i in range(min(5, feature_dir.shape[1])):
    w = feature_dir[:,i]
    order = np.argsort(np.abs(w))[::-1][:6]
    print(f"IC{i+1} top features:")
    for j in order:
        print(f"  {wine.feature_names[j]:25s} {w[j]:+.3f}")

## 7) Save Artifacts & Download

In [ ]:
import os, json as _json
os.makedirs("artifacts", exist_ok=True)

art = {
    "kde": {"h_1d": float(h_best)},
    "gmm": {"k_star": int(k_best), "pi": best_gmm["pi"].tolist(),
            "mu": best_gmm["mu"].tolist(),
            "Sigma": [S.tolist() for S in best_gmm["Sigma"]]},
    "pca": {"cum_evr": np.cumsum(evr_all).tolist()},
    "ica": {"W": Wica.tolist()}
}
with open("artifacts/summary.json","w") as f:
    _json.dump(art, f, indent=2)

np.savez("artifacts/density_feature_models.npz",
         X2=X2, X2s=X2s, mu2=mu2, sd2=sd2,
         Xw=Xw, Xws=Xws, muw=muw, sdw=sdw,
         kde1d_h=h_best, gmm_pi=best_gmm["pi"], gmm_mu=best_gmm["mu"], gmm_Sigma=np.array(best_gmm["Sigma"], dtype=object),
         pca_components=Cw, pca_scores=Zw, pca_evr=evr_all,
         ica_sources=Sica, ica_W=Wica, ica_A=Aica)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 8) Notes & Extensions

- Adaptive bandwidth KDE; anisotropic bandwidth matrices.
- Diagonal/full-covariance options in GMM and robust initializations (k-means++).
- Scree test and parallel analysis for PCA; sparse PCA as an extension.
- ICA with alternative nonlinearities and deflationary updates.
